In [1]:
import torch
torch.backends.cudnn.benchmark = True

In [8]:
# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Currently using: {device}")

if device == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU available, using CPU.")

Currently using: cuda
NVIDIA GeForce RTX 4060 Laptop GPU


In [9]:
import pandas as pd

In [ ]:
def load_and_save_data(
    corpus_url: str, qrels_url: str, queries_url: str, path_prefix: str
):
    try:
        print(f"Loading corpus from {corpus_url}")
        df_corpus = pd.read_parquet(corpus_url, engine="pyarrow")

        print(f"Loading qrels from {qrels_url}")
        df_qrels = pd.read_parquet(qrels_url, engine="pyarrow")

        print(f"Loading queries from {queries_url}")
        df_queries = pd.read_parquet(queries_url, engine="pyarrow")

        # Save files
        print(f"Saving corpus ({len(df_corpus)} documents)")
        df_corpus.to_json(f"{path_prefix}/corpus.jsonl", orient="records", lines=True)

        print(f"Saving queries ({len(df_queries)} queries)")
        df_queries.to_json(f"{path_prefix}/queries.jsonl", orient="records", lines=True)

        print(f"Saving qrels ({len(df_qrels)} relevance judgments)")
        df_qrels.to_csv(f"{path_prefix}/qrels/test.tsv", sep="\t", index=False)

        return True
    except Exception as e:
        print(f"Error: {e}")
        return False

In [11]:
import yaml

def load_config(config_path: str):
    with open(config_path, "r") as file:
        config = yaml.safe_load(file)
    return config

In [ ]:
datasets = load_config("../configs/datasets.yaml")

# Create datasets directory if it doesn't exist
import os
from tqdm import tqdm

os.makedirs("../datasets", exist_ok=True)

# Process each dataset
successful_downloads = []
failed_downloads = []

for dataset_name, dataset_config in tqdm(datasets.items(), desc="Processing datasets"):
    print(f"\nProcessing dataset: {dataset_name}")

    # Create dataset-specific directory
    dataset_path = f"../datasets/{dataset_name}"
    os.makedirs(dataset_path, exist_ok=True)
    os.makedirs(f"{dataset_path}/qrels", exist_ok=True)
    # Extract URLs from config
    corpus_url = dataset_config.get("corpus_url")
    qrels_url = dataset_config.get("qrels_url")
    queries_url = dataset_config.get("queries_url")

    if all([corpus_url, qrels_url, queries_url]):
        success = load_and_save_data(corpus_url, qrels_url, queries_url, dataset_path)
        if success:
            successful_downloads.append(dataset_name)
            print(f"{dataset_name} completed successfully!")
        else:
            failed_downloads.append(dataset_name)
    else:
        missing = [
            k
            for k, v in [
                ("corpus_url", corpus_url),
                ("qrels_url", qrels_url),
                ("queries_url", queries_url),
            ]
            if not v
        ]
        print(f"Missing URLs for {dataset_name}: {missing}")
        failed_downloads.append(dataset_name)

# Summary
print(f"\nDownload Summary:")
print(f"Successful: {len(successful_downloads)} datasets")
print(f"Failed: {len(failed_downloads)} datasets")

if successful_downloads:
    print(f"Successfully downloaded: {', '.join(successful_downloads)}")
if failed_downloads:
    print(f"Failed to download: {', '.join(failed_downloads)}")

Processing datasets:   0%|          | 0/2 [00:00<?, ?it/s]


Processing dataset: Bharat_NanoMSMARCO
Loading corpus from hf://datasets/carlfeynman/Bharat_NanoMSMARCO_ml/corpus/train-00000-of-00001.parquet
Loading qrels from hf://datasets/carlfeynman/Bharat_NanoMSMARCO_ml/qrels/train-00000-of-00001.parquet
Loading queries from hf://datasets/carlfeynman/Bharat_NanoMSMARCO_ml/queries/train-00000-of-00001.parquet


Processing datasets:  50%|█████     | 1/2 [00:04<00:04,  4.17s/it]

Saving corpus (5043 documents)
Saving queries (50 queries)
Saving qrels (50 relevance judgments)
Bharat_NanoMSMARCO completed successfully!

Processing dataset: MTEBIndicQARetrieval
Loading corpus from https://huggingface.co/datasets/mteb/IndicQARetrieval/resolve/main/ml-corpus/test-00000-of-00001.parquet
Loading qrels from https://huggingface.co/datasets/mteb/IndicQARetrieval/resolve/main/ml-qrels/test-00000-of-00001.parquet
Loading queries from https://huggingface.co/datasets/mteb/IndicQARetrieval/resolve/main/ml-queries/test-00000-of-00001.parquet


Processing datasets: 100%|██████████| 2/2 [00:06<00:00,  3.34s/it]

Saving corpus (247 documents)
Saving queries (1587 queries)
Saving qrels (1587 relevance judgments)
MTEBIndicQARetrieval completed successfully!

Download Summary:
Successful: 2 datasets
Failed: 0 datasets
Successfully downloaded: Bharat_NanoMSMARCO, MTEBIndicQARetrieval
